In [1]:
import os
import time
import json
import math
import random
import numpy as np
import pandas as pd
from tqdm import tqdm
from typing import List, Dict, Optional

from sklearn.model_selection import train_test_split
from sklearn.metrics         import mean_absolute_error, r2_score, root_mean_squared_error

from openai import OpenAI

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

In [2]:
def row_to_prompt(row: pd.Series) -> str:
    skills = {', '.join([skill for skill, has_skill in zip(['Python', 'Spark', 'AWS'], [row['Python_yn'], row['Spark'], row['AWS_yn']]) if has_skill])}
    
    return (
        '**Company Information**: \n'
        f"- Company name: {row['Company Name'] or 'Not specified'}\n"
        f"- Company location: {row['Location'] or 'Not specified'}\n"
        f"- Company headquarters: {row['Headquarters'] or 'Not specified'}\n"
        f"- Company size: {row['Size'] or 'Not specified'}\n"
        f"- Type of ownership: {row['Type of ownership'] or 'Not specified'}\n"
        f"- Industry: {row['Industry'] or 'Not specified'}\n"
        f"- Sector: {row['Sector'] or 'Not specified'}\n"
        f"- Revenue: {row['Revenue'] or 'Not specified'}\n"
        
        '\n**Job Information**: \n'
        f"- Job title: {row['job_simplified'] or 'Not specified'}\n"
        f"- Job level: {row['seniority'] or 'Not specified'}\n"
        f"- Job state: {row['job_state'] or 'Not specified'}\n"
        f"- Job rating: {row['Rating Category'] or 'Not specified'}\n"
        f"- Skills requirement: {skills if skills else 'Not specified'}\n"
    )


def make_example(row: pd.Series) -> str:
    return f"{row_to_prompt(row)}\n**Annual Salary (in USD)**: {row['Average Salary']:.3f}"

In [3]:
def quantile_bins(y: pd.Series, q: int=4) -> pd.Series:
    """Robust bining: Handle duplicates/edges by ranking first."""
    return pd.qcut(y.rank(method='average', pct=True), q, labels=False, duplicates='drop')


def prepare_examples(df_train: pd.DataFrame, k: int) -> pd.DataFrame:
    """
    Strategy: Stratify by (salary_bin, job_simplified) to get coverage.
    Falls back to random if insufficient samples are available.
    """
    tmp          = df_train.copy()
    tmp['__bin'] = quantile_bins(tmp['Average Salary'], q=min(6, max(3,k)))

    groups = list(tmp.groupby(['__bin', 'job_simplified']))
    random.shuffle(groups)

    examples = []
    for _, group in groups:
        selected = group.sample(n=1, random_state=random.randint(0, 10_000)).iloc[0]
        examples.append(selected)

        if len(examples) >= k:
            break

    if len(examples) < k:
        remaining = tmp.drop(index=[row.name for row in examples])

        if len(remaining) >= k - len(examples):
            selected = list(remaining.sample(n=(k-len(examples)), random_state=42).itertuples(index=False))
            examples.extend(selected)

    return pd.DataFrame(examples).reset_index(drop=True)

In [4]:
def build_prompt(examples_df: pd.DataFrame, test_row: pd.Series) -> List[Dict[str, str]]:
    """Build a system + user style input for the response API."""
    
    # System prompt
    system = (
        'You are an expert salary prediction assistant. '
        'Given company and/or job information, predict the **Annual Salary** in USD. '
        'Use knowledge of typical compensation by role, seniority, company size, industry, and tech stack. '
        'When unsure, or lacking information, make a single best point estimate. '
        'DO NOT add commentary or explanations beyond the salary estimate. Only provide the salary amount in USD.'
    )
    
    # Few-shot examples & query
    examples_txt = '\n\n'.join(make_example(row) for _, row in examples_df.iterrows())
    query_txt    = row_to_prompt(test_row)
    
    # User prompt
    user = (
        'Here are labeled examples:\n'
        f"{examples_txt}\n\n"
        'Now predict for this case (return ONLY the number via the provided JSON schema):\n'
        f"{query_txt}"
        '**Annual Salary (in USD)**: '
    )
    
    return [
        {'role': 'system', 'content': system},
        {'role': 'user'  , 'content': user}
    ]

In [5]:
def call_openai_with_schema(messages: List[Dict[str, str]], client: OpenAI) -> Optional[float]:
    """Uses JSON schema structured output so the model must return: {'predicted_salary': <number>}"""

    backoff = 2.0
    for attempt in range(1, 5+1):
        try:
            resp = client.responses.create(
                model       = 'gpt-5-nano',
                input       = messages,
                # temperature = 0.1,
                text        = {
                    'format': {
                        'type'       : 'json_schema',
                        'name'       : 'salary_prediction',
                        'schema': {
                            'type'                : 'object',
                            'properties'          : {'predicted_salary': {'type': 'number'}},
                            'required'            : ['predicted_salary'],
                            'additionalProperties': False
                        },
                    }
                }
            )
            
            # With structured outputs, the SDK exposes parsed content via 'output_text', which should be a JSON string, or via 'output' tokens
            txt = getattr(resp, 'output_text', None)
            
            if not txt:
                # Fallback: assemble text from content parts
                parts = []
                for item in getattr(resp, "output", []):
                    for c in getattr(item, 'content', []):
                        if getattr(c, 'type', '') == 'output_text':
                            parts.append(c.text)
                txt = ''.join(parts) if parts else None

            if not txt:
                raise RuntimeError('Empty response from model')

            data = json.loads(txt)
            val  = float(data['predicted_salary'])
            
            # Basic sanity clamp to avoid outrageous hallucinations
            if not math.isfinite(val):
                raise ValueError('Non-finite value')
            return val
        
        except Exception as e:
            if attempt == 5:
                print(f"[ERROR] Final failure: {e}", flush=True)
                return None
            
            sleep_s = backoff * (1.0 + 0.25 * random.random())
            print(f"[WARN] API error ({e}). Retrying in {sleep_s:.1f}s...", flush=True)
            time.sleep(sleep_s)
            backoff *= 2.0
            
    return None

In [6]:
df = pd.read_csv('../data/data_model.csv')
df.head()

,Company Name,Location,Headquarters,Size,Type of ownership,Industry,Sector,Revenue,job_simplified,seniority,Rating Category,job_state,Python_yn,Spark,AWS_yn,Average Salary
0,Tecolote Research,"Albuquerque, NM","Goleta, CA",501 to 1000 employees,Company - Private,Aerospace & Defense,Aerospace & Defense,$50 to $100 million (USD),data scientist,Other,Medium Rating,NM,1,0,0,72000.0
1,University of Maryland Medical System,"Linthicum, MD","Baltimore, MD",10000+ employees,Other Organization,Health Care Services & Hospitals,Health Care,$2 to $5 billion (USD),data scientist,Other,Medium Rating,MD,1,0,0,87500.0
2,KnowBe4,"Clearwater, FL","Clearwater, FL",501 to 1000 employees,Company - Private,Security Services,Business Services,$100 to $500 million (USD),data scientist,Other,High Rating,FL,1,1,0,85000.0
3,PNNL,"Richland, WA","Richland, WA",1001 to 5000 employees,Government,Energy,"Oil, Gas, Energy & Utilities",$500 million to $1 billion (USD),data scientist,Other,Medium Rating,WA,1,0,0,76500.0
4,Affinity Solutions,"New York, NY","New York, NY",51 to 200 employees,Company - Private,Advertising & Marketing,Business Services,Unknown,data scientist,Other,Low Rating,NY,1,0,0,114500.0


In [7]:
# Train/test split (stratify by discretized salary for stability)
strat             = pd.qcut(df['Average Salary'].rank(pct=True), q=5, labels=False, duplicates='drop')
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42, stratify=strat)

print(f"Train size: {len(train_df)}, Test size: {len(test_df)}")

Train size: 593, Test size: 149


In [8]:
# Prepare few-shot examples
examples_df = prepare_examples(train_df, k=5)
examples_df

,Company Name,Location,Headquarters,Size,Type of ownership,Industry,Sector,Revenue,job_simplified,seniority,Rating Category,job_state,Python_yn,Spark,AWS_yn,Average Salary,__bin
0,Boys Town Hospital,"Omaha, NE","Omaha, NE",1001 to 5000 employees,Hospital,Health Care Services & Hospitals,Health Care,Unknown,research scientist,Other,Low Rating,NE,0,0,0,61500.0,0
1,C Space,"Boston, MA","Boston, MA",201 to 500 employees,Company - Public,Advertising & Marketing,Business Services,$100 to $500 million (USD),data scientist,Other,Medium Rating,MA,1,0,0,56500.0,0
2,Palermo's Pizza,"Milwaukee, WI","Milwaukee, WI",501 to 1000 employees,Company - Private,Food & Beverage Manufacturing,Manufacturing,Unknown,software engineer,Other,Medium Rating,WI,0,0,0,54000.0,0
3,MassMutual,"Boston, MA","Springfield, MA",5001 to 10000 employees,Company - Private,Insurance Carriers,Insurance,$10+ billion (USD),other,Other,Medium Rating,MA,1,1,1,129500.0,3
4,Credit Sesame,"Mountain View, CA","Mountain View, CA",51 to 200 employees,Company - Private,Internet,Information Technology,$50 to $100 million (USD),data engineer,Senior/Principal,High Rating,CA,1,1,1,205000.0,4


In [9]:
# Predict on test set
client = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))
preds, gts, ids = [], [], []

for i, row in tqdm(test_df.reset_index(drop=True).iterrows(), total=len(test_df), desc='Predicting'):
    prompt = build_prompt(examples_df, row)
    y_pred = call_openai_with_schema(prompt, client)
    y_true = float(row['Average Salary'])
    
    preds.append(y_pred if y_pred is not None else np.nan)
    gts.append(y_true)
    ids.append(i)

results_df = pd.DataFrame({'id': ids, 'prediction': preds, 'ground_truth': gts})

print(f"RMSE: {root_mean_squared_error(results_df['ground_truth'], results_df['prediction']):.3f}")
print(f"MAE : {mean_absolute_error(results_df['ground_truth'], results_df['prediction']):.3f}")
print(f"R2  : {r2_score(results_df['ground_truth'], results_df['prediction']):.3f}")

Predicting: 100%|██████████| 149/149 [16:56<00:00,  6.82s/it]

RMSE: 45441.561
MAE : 36626.846
R2  : -0.460
